In [17]:
# !pip install -q vllm

In [18]:
# !pip install pyngrok

(APIServer pid=876) INFO 07-26 09:01:47 [loggers.py:310] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 8.5 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 16.3%
(APIServer pid=876) INFO 07-26 09:01:57 [loggers.py:310] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 16.3%
(APIServer pid=876) INFO:     129.213.83.121:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=876) INFO:     129.213.83.121:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=876) INFO 07-26 09:03:17 [loggers.py:310] Engine 000: Avg prompt throughput: 16.4 tokens/s, Avg generation throughput: 5.5 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 39.4%
(APIServer pid=876) INFO:     129.213.83.121:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer p

In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [7]:
# import os
# import torch
# import gc

# os.system("pkill -9 -f vllm")
# os.system("pkill -9 -f python")

# gc.collect()
# torch.cuda.empty_cache()
# print("🧹Clean GPU Before Starting to Free VRAM")

In [10]:
import subprocess
import time

# vLLM OpenAI-Compatible Server in Background
server_process = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "A7med-Ame3/Qwen2.5-7B-LiveKit-16bit",
    "--tensor-parallel-size", "2",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.80",
    "--enforce-eager", 
    "--port", "8000"
])

print("vLLM Server Loading.....")
time.sleep(40)  # Enough 

vLLM Server Loading.....
(APIServer pid=876) INFO 07-26 08:56:33 [api_utils.py:345] 
(APIServer pid=876) INFO 07-26 08:56:33 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=876) INFO 07-26 08:56:33 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.26.0
(APIServer pid=876) INFO 07-26 08:56:33 [api_utils.py:345]   █▄█▀ █     █     █     █  model   A7med-Ame3/Qwen2.5-7B-LiveKit-16bit
(APIServer pid=876) INFO 07-26 08:56:33 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=876) INFO 07-26 08:56:33 [api_utils.py:345] 
(APIServer pid=876) INFO 07-26 08:56:33 [api_utils.py:273] non-default args: {'model': 'A7med-Ame3/Qwen2.5-7B-LiveKit-16bit', 'max_model_len': 4096, 'enforce_eager': True, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.8}


(APIServer pid=876) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(APIServer pid=876) INFO 07-26 08:56:34 [model.py:623] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=876) INFO 07-26 08:56:34 [model.py:1788] Using max model len 4096
(APIServer pid=876) INFO 07-26 08:56:34 [vllm.py:1109] Asynchronous scheduling is enabled.
(APIServer pid=876) WARNING 07-26 08:56:34 [vllm.py:1163] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(APIServer pid=876) WARNING 07-26 08:56:34 [vllm.py:1213] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(APIServer pid=876) INFO 07-26 08:56:34 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(APIServer pid=876) INFO 07-26 08:56:34 [vllm.py:1392] Cudagraph is disabled under eager mode
(APIServer pid=876) INFO 07-26 08:56:34 [compilation.p

In [13]:
from pyngrok import ngrok
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Make vLLM run on Public Link
public_url = ngrok.connect(8000)
print("🔗 Public Link:")
print(public_url.public_url)

🔗 Public Link:                                                                                     
https://unsignalised-englacial-vinnie.ngrok-free.dev
(APIServer pid=876) INFO:     156.198.11.117:0 - "GET /v1/models HTTP/1.1" 200 OK
(APIServer pid=876) INFO:     156.198.11.117:0 - "GET /v1/models HTTP/1.1" 200 OK


In [14]:
import time
from openai import OpenAI

server_url = "http://localhost:8000"
model_name = "A7med-Ame3/Qwen2.5-7B-LiveKit-16bit"
prompt = "ازيك؟ عامل ايه النهار ده؟"

client = OpenAI(base_url=f"{server_url}/v1", api_key="not-needed")

start_time = time.time()
response = client.completions.create(
    model=model_name, 
    prompt=prompt,
    max_tokens=128,
    temperature=0.7,
)
end_time = time.time()

# Extract response
response_text = response.choices[0].text
latency = end_time - start_time

# --- RESPONSE ---
print(f"\n--- RESPONSE ---")
print(f"Model: {response.model}")
print(f"Response: {response_text[:200]}")
print(f"Latency: {latency:.2f}s")

if response.usage:
    print(f"Prompt tokens: {response.usage.prompt_tokens}")
    print(f"Completion tokens: {response.usage.completion_tokens}")

# --- API DETAILS ---
print(f"\n--- API DETAILS ---")
print(f"Endpoint: {server_url}/v1/completions")
print(f"Format: OpenAI-compatible (drop-in replacement)")
print(f"Auth: No API key needed (local server)")

# --- KEY INSIGHT ---
print("\n" + "=" * 65)
print("KEY INSIGHT:")
print("- vLLM serves an OpenAI-compatible API out of the box")
print("- Any app using the OpenAI SDK works with vLLM - zero code changes")
print("- This is how you self-host LLMs in production")
print("=" * 65)

(Worker_TP0 pid=933) WARNING 07-26 09:01:21 [jit_monitor.py:135] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=933) WARNING 07-26 09:01:23 [jit_monitor.py:135] Triton kernel JIT compilation during inference: reduce_segments. This causes a latency spike; consider extending warmup to cover this shape/config.
(APIServer pid=876) INFO 07-26 09:01:27 [loggers.py:310] Engine 000: Avg prompt throughput: 1.3 tokens/s, Avg generation throughput: 5.5 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%

--- RESPONSE ---(APIServer pid=876) INFO:     127.0.0.1:38808 - "POST /v1/completions HTTP/1.1" 200 OK

Model: A7med-Ame3/Qwen2.5-7B-LiveKit-16bit
Response: 
الليلة دي بتعتبر من الليالي المباركة والمعفاة عن الصيام، واللي بيعمل فيها الخير بيتقبل منه أضعاف مضاعفة. كتير من الناس بيقضوا ليلة النصف من شوال في صلاة التوبة أو قراءة ال

In [15]:
def ask_shekish(user_prompt: str, max_tokens: int = 512, temperature: float = 0.7, show_logs: bool = True) -> str:

    start_time = time.time()
    
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "انت شيخ مصري تجيب على اسئلة الناس فى الدين الاسلامى باللهجة المصرية"},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    
    end_time = time.time()
    
    answer = response.choices[0].message.content
    latency = end_time - start_time
    
    if show_logs:
        print(f"\n⏱️Inference Time: {latency:.2f} Seconds")
        if response.usage:
            print(f"📊 Prompt Tokens: {response.usage.prompt_tokens} | Completion Tokens: {response.usage.completion_tokens}")
            
    return answer

In [16]:
reply = ask_shekish("ما الفرق بين النبي و الرسول؟")
print("Response:\n", reply)

print("-"*20) 

reply_2 = ask_shekish("ما هو الاحسان ؟", show_logs=False)
print("Response:\n", reply_2)

(APIServer pid=876) INFO:     127.0.0.1:38808 - "POST /v1/chat/completions HTTP/1.1" 200 OK

⏱️Inference Time: 5.06 Seconds
📊 Prompt Tokens: 44 | Completion Tokens: 88
Response:
 النبي هو اللي ربنا بعته عشان يوصل رسالته للناس، بس ممكن يكون في رسالة قبل كده. أما الرسول فده أعلى مرتبة، ربنا بعته برسالة جديدة أو بسّر رسالة نبي قبله، وكمان جاب معاه تشريع جديد أو قانون. يعني كل رسول نبي لكن مش كل نبي رسول.
--------------------
(APIServer pid=876) INFO 07-26 09:01:37 [loggers.py:310] Engine 000: Avg prompt throughput: 6.9 tokens/s, Avg generation throughput: 17.0 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 16.3%
(APIServer pid=876) INFO:     127.0.0.1:38808 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Response:
 الاحسان هو إنك تعمل كل حاجة بإتقان وجمال، سواء كانت عبادة لربنا أو معاملة مع الناس. يعني إنك تحسن عملك كأنك شايف ربنا بيحكم عليك، أو على الأقل تكون متيقن إنه شايفك ومطلع عليك. ده بيشمل إنك تكون صادق وأمين، وتتعامل برحمة وعدل مع اللي ح